# 4.4 End-to-End ML Pipeline> Combine everything: data loading → feature engineering → training → evaluation → saving.> 💡 This is where your data engineering background gives you an edge over pure ML practitioners.---

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.pipeline import Pipelinefrom sklearn.compose import ColumnTransformerfrom sklearn.preprocessing import StandardScaler, OneHotEncoderfrom sklearn.impute import SimpleImputerfrom sklearn.ensemble import GradientBoostingClassifierfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.metrics import classification_reportfrom dataclasses import dataclassimport json@dataclassclass PipelineConfig:    """Configuration for the ML pipeline."""    test_size: float = 0.2    random_state: int = 42    n_estimators: int = 100    target_column: str = "readmitted"    numeric_features: list = None    categorical_features: list = Nonedef create_sample_data(n: int = 1000) -> pd.DataFrame:    """Generate sample healthcare data."""    np.random.seed(42)    df = pd.DataFrame({        "age": np.random.randint(18, 90, n),        "gender": np.random.choice(["M", "F"], n),        "diagnosis": np.random.choice(["diabetes", "copd", "heart_failure", "pneumonia"], n),        "length_of_stay": np.random.exponential(5, n).round(1),        "num_procedures": np.random.poisson(2, n),        "num_medications": np.random.poisson(5, n),        "total_cost": np.random.lognormal(8, 1, n).round(2),    })    # Simulate readmission target    risk = (0.01 * df["age"] + 0.1 * df["length_of_stay"] +             0.05 * df["num_medications"] + np.random.randn(n) * 2)    df["readmitted"] = (risk > 5).astype(int)    return dfdef build_pipeline(config: PipelineConfig) -> Pipeline:    """Build the full ML pipeline."""    preprocessor = ColumnTransformer([        ("num", Pipeline([            ("imputer", SimpleImputer(strategy="median")),            ("scaler", StandardScaler()),        ]), config.numeric_features),        ("cat", Pipeline([            ("imputer", SimpleImputer(strategy="most_frequent")),            ("encoder", OneHotEncoder(drop="first", sparse_output=False)),        ]), config.categorical_features),    ])        return Pipeline([        ("preprocessor", preprocessor),        ("classifier", GradientBoostingClassifier(            n_estimators=config.n_estimators,            random_state=config.random_state,        )),    ])# Run the pipelineconfig = PipelineConfig(    numeric_features=["age", "length_of_stay", "num_procedures", "num_medications", "total_cost"],    categorical_features=["gender", "diagnosis"],)df = create_sample_data()X = df.drop(columns=[config.target_column])y = df[config.target_column]X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=config.test_size, random_state=config.random_state)pipeline = build_pipeline(config)# Cross-validationcv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1")print(f"CV F1 scores: {cv_scores.round(3)}")print(f"Mean F1: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")# Final training and evaluationpipeline.fit(X_train, y_train)y_pred = pipeline.predict(X_test)print(f"\n{classification_report(y_test, y_pred, target_names=['No Readmit', 'Readmit'])}")